# 0. Setup & Installation
Run this cell first. Restart runtime if prompted.

In [ ]:
!pip install -U transformers peft bitsandbytes accelerate datasets trl scikit-learn --quiet

# 1. Authentication

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
secret_value_1 = user_secrets.get_secret("HF_TOKEN")
secret_value_2 = user_secrets.get_secret("OpenAI_API")

login(token=secret_value_1)

# 2. Configuration

In [ ]:
from dataclasses import dataclass, field
from typing import List


@dataclass
class ExperimentConfig:
    # ---- Model ----
    model_name: str = "Qwen/Qwen3-4B"
    max_seq_length: int = 2048

    # ---- LoRA ----
    lora_r: int = 8
    lora_alpha: int = 16
    lora_dropout: float = 0.05
    lora_target_modules: List[str] = field(default_factory=lambda: [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ])

    # ---- Training ----
    num_train_epochs: int = 3
    per_device_train_batch_size: int = 1
    gradient_accumulation_steps: int = 8
    learning_rate: float = 2e-4
    warmup_ratio: float = 0.1

    # ---- Quantization ----
    load_in_4bit: bool = True
    bnb_4bit_quant_type: str = "nf4"
    use_double_quant: bool = True

    # ---- BCE ----
    bce_weight: float = 1.0

    # ---- Special tokens ----
    prediction_start_token: str = "<|prediction|>"
    prediction_end_token: str = "<|/prediction|>"

    # ---- Paths ----
    output_dir: str = "./outputs"


cfg = ExperimentConfig()
print(f"Config: {cfg}")

# 3. Define Special Tokens

In [ ]:
SPECIAL_TOKENS = [
    # Input structure
    "<|system|>", "<|/system|>",
    "<|conversation|>", "<|/conversation|>",
    "<|agent|>", "<|customer|>",
    # Output structure
    "<|prediction|>", "<|/prediction|>",
]

# 4. Load Training Data

In [ ]:
import pandas as pd
from datasets import Dataset

train_data = pd.read_csv("/kaggle/input/notebooks/zygong1994/finetuning-experiment-0326-etl/train.csv")
eval_data = pd.read_csv("/kaggle/input/notebooks/zygong1994/finetuning-experiment-0326-etl/test.csv")

train_dataset = Dataset.from_pandas(train_data[["text"]])
eval_dataset = Dataset.from_pandas(eval_data[["text"]])

print(f"Train: {len(train_dataset)} samples")
print(f"Eval:  {len(eval_dataset)} samples")

# 5. Load Model + Tokenizer (QLoRA)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# ---- Quantization config ----
bnb_config = BitsAndBytesConfig(
    load_in_4bit=cfg.load_in_4bit,
    bnb_4bit_quant_type=cfg.bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=cfg.use_double_quant,
)

# ---- Load tokenizer ----
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

# ---- Add special tokens ----
num_added = tokenizer.add_special_tokens({
    "additional_special_tokens": SPECIAL_TOKENS
})
print(f"Added {num_added} special tokens. Vocab size: {len(tokenizer)}")

# ---- Get critical token IDs for BCE loss ----
prediction_token_id = tokenizer.convert_tokens_to_ids("<|prediction|>")
label_0_token_id = tokenizer.convert_tokens_to_ids("0")
label_1_token_id = tokenizer.convert_tokens_to_ids("1")

print(f"Token IDs -> <|prediction|>: {prediction_token_id}, '0': {label_0_token_id}, '1': {label_1_token_id}")
assert label_0_token_id != tokenizer.unk_token_id, "'0' mapped to UNK!"
assert label_1_token_id != tokenizer.unk_token_id, "'1' mapped to UNK!"

# ---- Load model ----
model = AutoModelForCausalLM.from_pretrained(
    cfg.model_name,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa",
    torch_dtype=torch.bfloat16
)

# ---- Resize embeddings for new special tokens ----
model.resize_token_embeddings(len(tokenizer))
print(f"Model embeddings resized to {len(tokenizer)}")

# 6. Apply LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=cfg.lora_r,
    lora_alpha=cfg.lora_alpha,
    target_modules=cfg.lora_target_modules,
    modules_to_save=["embed_tokens", "lm_head"],
    lora_dropout=cfg.lora_dropout,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

# 7. BCE + SFT Dual-Loss Trainer

In [ ]:
import torch.nn.functional as F
from torch.nn import BCEWithLogitsLoss
from trl import SFTTrainer, SFTConfig


class BCESFTTrainer(SFTTrainer):
    """
    Extends SFTTrainer to add BCE loss on the <|prediction|> token position.

    How it works:
    1. Standard causal LM loss is computed over all tokens (inherited).
    2. We find the position of <|prediction|> in each sequence.
    3. The logit at that position for token "1" vs "0" is used to compute
       BCE loss against the ground-truth label.
    4. Total loss = lm_loss + bce_weight * bce_loss
    """

    def __init__(self, *args, bce_weight=1.0, prediction_token_id=None,
                 label_0_token_id=None, label_1_token_id=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.bce_weight = bce_weight
        self.prediction_token_id = prediction_token_id
        self.label_0_token_id = label_0_token_id
        self.label_1_token_id = label_1_token_id
        self.bce_loss_fn = BCEWithLogitsLoss()

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """
        Override compute_loss to add BCE on <|prediction|> token.

        In causal LM, the model predicts the NEXT token at each position.
        So if <|prediction|> is at position i, the logits at position i
        predict what comes at position i+1 (which is "0" or "1").
        """
        labels = inputs.get("labels")
        outputs = model(**inputs)
        lm_loss = outputs.loss
        logits = outputs.logits

        input_ids = inputs["input_ids"]
        batch_size = input_ids.size(0)

        bce_losses = []
        for b in range(batch_size):
            pred_positions = (input_ids[b] == self.prediction_token_id).nonzero(as_tuple=True)[0]

            if len(pred_positions) == 0:
                continue

            pred_pos = pred_positions[0].item()

            logit_0 = logits[b, pred_pos, self.label_0_token_id]
            logit_1 = logits[b, pred_pos, self.label_1_token_id]

            if pred_pos + 1 < input_ids.size(1):
                actual_next_token = input_ids[b, pred_pos + 1].item()
                if actual_next_token == self.label_1_token_id:
                    target = torch.tensor(1.0, device=logits.device)
                else:
                    target = torch.tensor(0.0, device=logits.device)

                bce_logit = logit_1 - logit_0
                bce_loss = self.bce_loss_fn(bce_logit, target)
                bce_losses.append(bce_loss)

        if bce_losses:
            bce_loss_mean = torch.stack(bce_losses).mean()
            total_loss = lm_loss + self.bce_weight * bce_loss_mean
        else:
            total_loss = lm_loss
            bce_loss_mean = torch.tensor(0.0, device=lm_loss.device)

        if self.state.global_step % self.args.logging_steps == 0:
            self.log({
                "lm_loss": lm_loss.detach().item(),
                "bce_loss": bce_loss_mean.detach().item(),
                "total_loss": total_loss.detach().item(),
            })

        return (total_loss, outputs) if return_outputs else total_loss

# 8. Training

In [ ]:
training_args = SFTConfig(
    output_dir=cfg.output_dir,
    num_train_epochs=cfg.num_train_epochs,
    per_device_train_batch_size=cfg.per_device_train_batch_size,
    gradient_accumulation_steps=cfg.gradient_accumulation_steps,
    learning_rate=cfg.learning_rate,
    warmup_ratio=cfg.warmup_ratio,
    lr_scheduler_type="cosine",
    fp16=False,
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    seed=42,
    report_to="none",
    max_length=cfg.max_seq_length,
    packing=False,
    dataset_text_field="text",
)

trainer = BCESFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    bce_weight=cfg.bce_weight,
    prediction_token_id=prediction_token_id,
    label_0_token_id=label_0_token_id,
    label_1_token_id=label_1_token_id,
)

print("Starting training...")
train_result = trainer.train()
print(f"\nTraining complete!")
print(f"Final train loss: {train_result.training_loss:.4f}")

# 9. Save LoRA Adapter

In [ ]:
adapter_path = f"{cfg.output_dir}/final_adapter"
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"Adapter saved to {adapter_path}")

import os
for f in os.listdir(adapter_path):
    size_mb = os.path.getsize(os.path.join(adapter_path, f)) / 1024 / 1024
    print(f"  {f}: {size_mb:.2f} MB")

import json
with open(os.path.join(adapter_path, "experiment_config.json"), "w") as f:
    json.dump(vars(cfg), f, indent=2)

# 10. Inference

In [ ]:
SYSTEM_PROMPT = (
    "You are an expert sales call analyst specializing in financial services. "
    "You will be given a full transcript of a customer-agent conversation. Your task is to:\n"
    "1. Predict whether the customer will convert (1) or not (0) based on signals in the conversation "
    "such as: buying intent, urgency, objections raised vs. resolved, competitor comparisons, "
    "explicit next-step requests, and overall engagement level.\n"
    "2. Provide a brief reasoning explaining the key signals that support your prediction."
)


def build_inference_prompt(system_prompt: str, transcript: str) -> str:
    return (
        f"<|system|>{system_prompt}<|/system|>\n"
        f"<|conversation|>\n{transcript}\n<|/conversation|>\n"
        f"<|prediction|>"
    )


def predict(model, tokenizer, transcript: str, max_new_tokens=200):
    prompt = build_inference_prompt(SYSTEM_PROMPT, transcript)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.pad_token_id,
        )

    new_tokens = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)
    return {
        "generated_output": new_tokens,
        "full_text": tokenizer.decode(outputs[0], skip_special_tokens=False),
    }


def predict_with_logit_probability(model, tokenizer, transcript: str):
    prompt = build_inference_prompt(SYSTEM_PROMPT, transcript)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits[:, -1, :]

    token_0 = tokenizer.encode("0", add_special_tokens=False)[0]
    token_1 = tokenizer.encode("1", add_special_tokens=False)[0]

    binary_logits = logits[:, [token_0, token_1]]
    probs = F.softmax(binary_logits, dim=-1)
    conversion_prob = probs[0, 1].item()

    return {
        "conversion_probability": conversion_prob,
        "predicted_class": 1 if conversion_prob > 0.5 else 0,
        "raw_logits": {"token_0": logits[0, token_0].item(), "token_1": logits[0, token_1].item()},
    }


# ---- Test on one example ----
test_row = eval_data.iloc[0]
test_transcript = test_row["full_text"]
test_label = test_row["outcome"]

print("=" * 60)
print("TEST INFERENCE: Full Generation")
print("=" * 60)
result = predict(model, tokenizer, test_transcript)
print(f"Ground truth: {test_label}")
print(f"Generated output:\n{result['generated_output']}")

print("\n" + "=" * 60)
print("TEST INFERENCE: Logit-Based Probability")
print("=" * 60)
prob_result = predict_with_logit_probability(model, tokenizer, test_transcript)
print(f"Ground truth: {test_label}")
print(f"Conversion probability: {prob_result['conversion_probability']:.4f}")
print(f"Predicted class: {prob_result['predicted_class']}")
print(f"Raw logits: {prob_result['raw_logits']}")

# 11. Evaluate on Full Eval Set

In [ ]:
from collections import defaultdict
from sklearn.metrics import roc_auc_score, classification_report

eval_results = []

print("Running evaluation...")
for i in range(len(eval_data)):
    row = eval_data.iloc[i]
    prob_result = predict_with_logit_probability(
        model, tokenizer, row["full_text"]
    )
    eval_results.append({
        "ground_truth": row["outcome"],
        "predicted_class": prob_result["predicted_class"],
        "probability": prob_result["conversion_probability"],
    })
    if (i + 1) % 10 == 0:
        print(f"  Processed {i+1}/{len(eval_data)} samples")

# ---- Compute metrics ----
correct = sum(1 for r in eval_results if r["ground_truth"] == r["predicted_class"])
total = len(eval_results)
accuracy = correct / total

y_true = [r["ground_truth"] for r in eval_results]
y_pred = [r["predicted_class"] for r in eval_results]
y_prob = [r["probability"] for r in eval_results]

print(f"\n{'='*60}")
print(f"EVALUATION RESULTS")
print(f"{'='*60}")
print(f"Accuracy: {correct}/{total} = {accuracy:.2%}")
print(f"AUC-ROC: {roc_auc_score(y_true, y_prob):.4f}")
print(f"\n{classification_report(y_true, y_pred, target_names=['No Convert (0)', 'Convert (1)'])}")